# Week 1 · Lab 1 — Markov Decision Processes from Scratch

**Reinforcement Learning (Graduate)**  ·  Reading: *Grokking Deep RL*, Ch. 1–2

---
**Name:** ______________________   **Date:** ____________

### What you'll do
1. Check your Python / PyTorch / Gymnasium setup.
2. Build three MDPs **by hand** as transition dictionaries: **Bandit Walk (BW)**, **Bandit Slippery Walk (BSW)**, and **Frozen Lake (4×4)**.
3. Verify every transition distribution is valid (probabilities sum to 1).
4. Confirm your Frozen Lake matches **Gymnasium**'s `FrozenLake-v1`.
5. Compute **returns** under a fixed policy and see how the **discount factor γ** changes them.

> **How to work:** fill in each cell marked `# TODO`, then run the **check** cell right below it. A check passes silently (no output / `assert` error) — if it raises an `AssertionError`, fix your code and re-run. Solutions are in the companion solution notebook.


## 1. Environment setup

Run the cell below. `numpy` and `gymnasium` are required for this lab. `torch` (PyTorch) isn't used until the deep-RL weeks (Week 7+), so it's fine if it isn't installed yet.

In [1]:
import importlib, sys
print("Python:", sys.version.split()[0])

for pkg in ["numpy", "gymnasium"]:
    assert importlib.util.find_spec(pkg) is not None, f"Required package '{pkg}' is not installed. Run: pip install {pkg}"

import numpy as np
import gymnasium as gym
print("numpy:    ", np.__version__)
print("gymnasium:", gym.__version__)

try:
    import torch
    print("torch:    ", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("torch:     not installed (only needed from Week 7 onward)")

print("\nSetup looks good." )

Python: 3.10.12
numpy:     2.2.6
gymnasium: 1.3.0
torch:     not installed (only needed from Week 7 onward)

Setup looks good.


## 2. How we represent an MDP

We store an MDP as a nested dictionary `P` with the same shape Gymnasium uses:

```
P[state][action] = [ (probability, next_state, reward, done), ... ]
```

For each `(state, action)` we list every possible outcome. The `probability` values for one `(state, action)` **must sum to 1**. `done` is `True` when `next_state` is terminal (a hole or the goal).

This is the *model* of the environment from Lecture B: the **transition function** (where you land) and the **reward function** (what you get), bundled together.

## 3. Exercise 1 — Bandit Walk (BW)

A one-row walk with **3 states**:

```
[ 0 (hole) ] — [ 1 (start) ] — [ 2 (goal) ]
```

- Actions: `0 = left`, `1 = right`.
- **Deterministic**: you always move where you intend.
- Reward is `+1` **only** for reaching the goal (state 2); everything else is `0`.
- States `0` and `2` are **terminal** — model them as self-loops with reward `0` and `done=True`.

Implement `build_bw()` so it returns the `P` dictionary.

In [2]:
def build_bw():
    """Bandit Walk MDP as P[s][a] = [(prob, next_state, reward, done), ...]."""
    P = {
        0: {  # left hole — terminal, absorbing
            0: [(1.0, 0, 0.0, True)],
            1: [(1.0, 0, 0.0, True)],
        },
        1: {  # start
            0: [(1.0, 0, 0.0, True)],   # left  -> hole
            1: [(1.0, 2, 1.0, True)],   # right -> goal (+1)
        },
        2: {  # goal — terminal, absorbing
            0: [(1.0, 2, 0.0, True)],
            1: [(1.0, 2, 0.0, True)],
        },
    }
    return P

P_bw = build_bw()

In [3]:
# --- check: Bandit Walk ---
def probs_sum_to_one(P):
    for s, actions in P.items():
        for a, transitions in actions.items():
            total = sum(p for (p, _, _, _) in transitions)
            assert abs(total - 1.0) < 1e-9, f"P[{s}][{a}] probabilities sum to {total}, not 1"

probs_sum_to_one(P_bw)
assert set(P_bw.keys()) == {0, 1, 2}
assert P_bw[1][1] == [(1.0, 2, 1.0, True)], "right from start should reach the goal with +1"
assert P_bw[1][0] == [(1.0, 0, 0.0, True)], "left from start should fall in the hole"
assert P_bw[0][0][0][3] is True and P_bw[2][1][0][3] is True, "states 0 and 2 must be terminal"
print("Bandit Walk OK")

Bandit Walk OK


## 4. Exercise 2 — Bandit Slippery Walk (BSW)

Same 3-state layout, but now the ground is **slippery**: you move where you intend with probability `p` (default `0.8`), and in the **opposite** direction with probability `1 - p`.

Implement `build_bsw(p=0.8)`. Each non-terminal `(state, action)` should now have **two** outcomes whose probabilities sum to 1.

In [4]:
def build_bsw(p=0.8):
    """Bandit Slippery Walk: intended direction w.p. p, opposite w.p. 1-p."""
    q = 1.0 - p
    P = {
        0: {0: [(1.0, 0, 0.0, True)], 1: [(1.0, 0, 0.0, True)]},
        1: {
            # left intended -> hole (p); slip right -> goal (q)
            0: [(p, 0, 0.0, True), (q, 2, 1.0, True)],
            # right intended -> goal (p); slip left -> hole (q)
            1: [(p, 2, 1.0, True), (q, 0, 0.0, True)],
        },
        2: {0: [(1.0, 2, 0.0, True)], 1: [(1.0, 2, 0.0, True)]},
    }
    return P

P_bsw = build_bsw()

In [5]:
# --- check: Bandit Slippery Walk ---
probs_sum_to_one(P_bsw)
assert len(P_bsw[1][1]) == 2, "slippery: right action should have two possible outcomes"
# probability of actually reaching the goal when choosing 'right'
p_goal = sum(p for (p, ns, r, d) in P_bsw[1][1] if ns == 2)
assert abs(p_goal - 0.8) < 1e-9, f"expected 0.8 chance of reaching goal, got {p_goal}"
print("Bandit Slippery Walk OK")

Bandit Slippery Walk OK


## 5. Exercise 3 — Frozen Lake (4×4), from scratch

The classic 4×4 map (`S`=start, `F`=frozen, `H`=hole, `G`=goal):

```
S F F F
F H F H
F F F H
H F F G
```

States are numbered `0..15` left-to-right, top-to-bottom. Actions follow Gymnasium: `0=LEFT, 1=DOWN, 2=RIGHT, 3=UP`.

**Slippery dynamics:** when you pick an action, you actually move in that direction **or** in one of the two **perpendicular** directions, each with probability **1/3**. Moving into a wall keeps you in place. Reward is `+1` only on entering the goal (state 15). Holes and the goal are terminal.

The grid constants and a deterministic `next_cell(state, action)` helper are given. Use them to implement `build_frozen_lake()`.

In [6]:
# --- given: Frozen Lake grid + deterministic move helper ---
NROW, NCOL = 4, 4
HOLES = {5, 7, 11, 12}
GOAL = 15
TERMINALS = HOLES | {GOAL}
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

def next_cell(state, action):
    """Deterministic move on the 4x4 grid, clamped at the walls."""
    row, col = divmod(state, NCOL)
    if action == LEFT:   col = max(col - 1, 0)
    elif action == DOWN: row = min(row + 1, NROW - 1)
    elif action == RIGHT:col = min(col + 1, NCOL - 1)
    elif action == UP:   row = max(row - 1, 0)
    return row * NCOL + col

In [7]:
def build_frozen_lake():
    """4x4 slippery Frozen Lake as P[s][a] = [(prob, next_state, reward, done), ...]."""
    P = {}
    for s in range(NROW * NCOL):
        P[s] = {}
        for a in range(4):
            if s in TERMINALS:
                # terminal states are absorbing
                P[s][a] = [(1.0, s, 0.0, True)]
                continue
            outcomes = []
            # intended action and the two perpendicular directions, each w.p. 1/3
            for b in [(a - 1) % 4, a, (a + 1) % 4]:
                ns = next_cell(s, b)
                reward = 1.0 if ns == GOAL else 0.0
                done = ns in TERMINALS
                outcomes.append((1.0 / 3.0, ns, reward, done))
            P[s][a] = outcomes
    return P

P_fl = build_frozen_lake()

In [8]:
# --- check: Frozen Lake is a valid MDP ---
probs_sum_to_one(P_fl)
assert set(P_fl.keys()) == set(range(16))
assert P_fl[12][RIGHT] == [(1.0, 12, 0.0, True)], "hole at 12 must be terminal"
# from start (0), choosing DOWN can land in {0 (slip left into wall), 4 (down), 1 (slip right)}
landing = {ns for (_, ns, _, _) in P_fl[0][DOWN]}
assert landing == {0, 1, 4}, f"unexpected landing cells from state 0 / DOWN: {landing}"
print("Frozen Lake structure OK")

Frozen Lake structure OK


## 6. Does it match Gymnasium?

Gymnasium ships the same MDP in `env.unwrapped.P`. We compare **distributions** (aggregating probability per `(next_state, reward, done)`), so ordering and duplicate entries don't matter.

In [9]:
import math

def distribution(transitions):
    """Aggregate a transition list into {(next_state, reward, done): total_prob}."""
    d = {}
    for (p, ns, r, done) in transitions:
        key = (ns, float(r), bool(done))
        d[key] = d.get(key, 0.0) + p
    return d

env = gym.make("FrozenLake-v1", is_slippery=True, map_name="4x4")
gymP = env.unwrapped.P

mismatch = 0
for s in range(16):
    for a in range(4):
        ours, theirs = distribution(P_fl[s][a]), distribution(gymP[s][a])
        keys = set(ours) | set(theirs)
        for k in keys:
            if not math.isclose(ours.get(k, 0.0), theirs.get(k, 0.0), abs_tol=1e-9):
                mismatch += 1

assert mismatch == 0, f"{mismatch} (state,action) outcomes differ from Gymnasium"
print("Our Frozen Lake matches Gymnasium's FrozenLake-v1 exactly.")
env.close()

Our Frozen Lake matches Gymnasium's FrozenLake-v1 exactly.


Let's also *interact* with the Gymnasium environment directly — reset it and take a few random steps, the way an agent would.

In [10]:
env = gym.make("FrozenLake-v1", is_slippery=True, map_name="4x4")
obs, info = env.reset(seed=0)
print("start state:", obs)

rng = np.random.default_rng(0)
for t in range(5):
    action = int(rng.integers(4))
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"t={t}  action={action}  -> state={obs}  reward={reward}  done={terminated or truncated}")
    if terminated or truncated:
        print("episode ended"); break
env.close()

start state: 0
t=0  action=3  -> state=1  reward=0  done=False
t=1  action=2  -> state=5  reward=0  done=True
episode ended


## 7. Returns under a fixed policy

The **return** from time $t$ is the discounted sum of future rewards:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

### Exercise 4 — `compute_return`
Given a list of rewards `[r0, r1, r2, ...]` (in time order) and a discount `gamma`, return $\sum_k \gamma^k r_k$.

In [11]:
def compute_return(rewards, gamma):
    """Discounted return of a reward sequence."""
    G = 0.0
    for k, r in enumerate(rewards):
        G += (gamma ** k) * r
    return G

In [12]:
# --- check: compute_return ---
assert abs(compute_return([1, 1, 1], 1.0) - 3.0) < 1e-9
assert abs(compute_return([1, 1, 1], 0.5) - 1.75) < 1e-9     # 1 + 0.5 + 0.25
assert abs(compute_return([0, 0, 5], 0.9) - 0.81 * 5) < 1e-9 # only the last reward, discounted twice
assert abs(compute_return([], 0.9)) < 1e-9
print("compute_return OK")

compute_return OK


### Exercise 5 — Monte-Carlo value of a state

We don't have value-iteration yet (that's Week 2), so we **estimate** the value of a state by simulation: run many episodes under a fixed policy and average the returns.

A `run_episode` helper is given (it samples outcomes from `P`). Implement `mc_value`, which averages `compute_return` over `n_episodes`.

In [13]:
# --- given: simulate one episode under a fixed (deterministic) policy ---
def run_episode(P, policy, start, rng, max_steps=200):
    """Follow `policy` (dict/array: state -> action) from `start`; return the list of rewards."""
    s = start
    rewards = []
    for _ in range(max_steps):
        a = policy[s]
        transitions = P[s][a]
        probs = [p for (p, _, _, _) in transitions]
        idx = rng.choice(len(transitions), p=probs)
        p, ns, r, done = transitions[idx]
        rewards.append(r)
        s = ns
        if done:
            break
    return rewards

In [14]:
def mc_value(P, policy, start, gamma, n_episodes=20000, seed=0):
    """Monte-Carlo estimate of the value of `start` under `policy`."""
    rng = np.random.default_rng(seed)
    total = 0.0
    for _ in range(n_episodes):
        rewards = run_episode(P, policy, start, rng)
        total += compute_return(rewards, gamma)
    return total / n_episodes

In [15]:
# --- check: Monte-Carlo value on the bandit walks ---
# "always go right" policy
pi_right = {0: 1, 1: 1, 2: 1}

# Deterministic BW: going right from the start always wins -> value 1.0 for any gamma
assert abs(mc_value(P_bw, pi_right, 1, gamma=0.9) - 1.0) < 1e-9, "BW start value should be exactly 1.0"

# Slippery BSW: right reaches goal w.p. 0.8 in one step -> expected return ~ 0.8
v_bsw = mc_value(P_bsw, pi_right, 1, gamma=0.9, n_episodes=40000)
assert abs(v_bsw - 0.8) < 0.02, f"BSW start value should be ~0.8, got {v_bsw:.3f}"
print(f"MC value OK  (BW=1.0, BSW={v_bsw:.3f})")

MC value OK  (BW=1.0, BSW=0.801)


### How the discount factor changes the return

Now the payoff: estimate the value of the **start state** of Frozen Lake under a fixed (hand-written) policy, sweeping `gamma` from `0` to `1`. Because the only reward is at the distant goal, a small `gamma` makes that far-off reward almost worthless, while a `gamma` near 1 values it — so the start-state value **grows with γ**.

In [16]:
# A fixed, goal-oriented policy (not necessarily optimal) for the 4x4 lake.
# actions: 0=LEFT 1=DOWN 2=RIGHT 3=UP
fixed_policy = {
    0: DOWN, 1: RIGHT, 2: DOWN,  3: LEFT,
    4: DOWN, 5: LEFT,  6: DOWN,  7: LEFT,
    8: RIGHT,9: DOWN,  10: DOWN, 11: LEFT,
    12: LEFT,13: RIGHT,14: RIGHT,15: LEFT,
}

print(f"{'gamma':>6} | {'value of start state':>20}")
print("-" * 31)
prev = -1.0
for gamma in [0.0, 0.5, 0.9, 0.95, 0.99, 1.0]:
    v = mc_value(P_fl, fixed_policy, 0, gamma=gamma, n_episodes=20000, seed=1)
    print(f"{gamma:>6.2f} | {v:>20.4f}")
    prev = v

print()
print("Notice: larger gamma -> the far-away goal reward is worth more today.")

 gamma | value of start state
-------------------------------


  0.00 |               0.0000


  0.50 |               0.0003


  0.90 |               0.0162


  0.95 |               0.0263


  0.99 |               0.0389


  1.00 |               0.0430

Notice: larger gamma -> the far-away goal reward is worth more today.


## 8. Wrap-up

You built three MDPs by hand, checked they're valid, matched Gymnasium, and computed returns under a fixed policy — seeing first-hand how **γ** trades off immediate vs. long-term reward.

**Next week (Week 2):** instead of *estimating* values by simulation, you'll *compute* them exactly with **dynamic programming** — policy evaluation, policy iteration, and value iteration — on these very environments.

> **Submit:** run all cells top-to-bottom (Kernel → Restart & Run All), make sure every check prints OK with no errors, then export/share this notebook.